# Multilingual Text Detoxification with Transformer Seq2Seq Models

## Team: Iman Chantieva, Abdul-Malik Khatuev, Irina Brodskaya, Akmuhammet Gurbangeldiyev

## Research goal
Given a toxic sentence in a language $L$, generate a non-toxic sentence in the same language while preserving meaning and fluency.

$$
x_{toxic} \rightarrow y_{neutral}
$$

## Main components
1. Data loading and exploratory analysis.
2. Rule-based detoxification baseline.
3. Pretrained multilingual seq2seq inference baseline.
4. Optional supervised fine-tuning / LoRA fine-tuning of mT5.
5. Candidate generation and multi-objective reranking.
6. Automatic evaluation: toxicity reduction, semantic preservation, fluency proxy, and per-language analysis.
7. Manual error analysis.



## 0. Project rationale

Text detoxification is a controlled text generation task. Unlike simple toxic comment classification, the system must **generate** a new sequence. Therefore, the core model should be an encoder-decoder or text-to-text transformer such as mT5, mBART, or a detoxification-specialized seq2seq model.

We evaluate the system along three axes:

- **Toxicity reduction**: the output should be less toxic than the input.
- **Meaning preservation**: the output should retain the original content.
- **Fluency**: the output should remain natural and grammatical.

This turns the original competition pipeline into a deep learning experiment with baselines, model adaptation, ablations, and error analysis.

## 1. Environment setup

In [5]:
# Uncomment if running in a fresh Colab/Kaggle environment.
!pip install -q datasets transformers sentencepiece accelerate evaluate sacrebleu sentence-transformers peft bitsandbytes tqdm jieba scikit-learn

In [6]:
import os
# Reduce CUDA memory fragmentation. This must be set before heavy CUDA allocations.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re
import math
import random
import gc
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd

import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
tqdm.pandas()

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def clear_cuda_memory():
    """Safely release cached CUDA memory between large model blocks."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

seed_everything(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total GPU memory: {total_gb:.2f} GB")


Device: cuda
GPU: Tesla T4
Total GPU memory: 14.56 GB


In [7]:
# Global project configuration.

DATASET_CANDIDATES = [
    "textdetox/multilingual_paradetox",        # preferred if available
    "textdetox/multilingual_paradetox_train",  # alternative naming
    "textdetox/multilingual_paradetox_test",   # original assignment source; may not contain references
]

LOCAL_DATA_PATH = None  # Example: "/content/train.csv" or "/kaggle/input/data/train.tsv"

TEXT_COL_CANDIDATES = ["toxic_sentence", "toxic_text", "toxic", "source", "text", "sentence"]
REF_COL_CANDIDATES = ["neutral_sentence", "neutral_text", "detoxified_text", "target", "reference", "neutral"]
LANG_COL_CANDIDATES = ["lang", "language", "language_code"]

BASE_SEQ2SEQ_MODEL = "google/mt5-small"

# Pretrained seq2seq baselines.
# `s-nlp/mt0-xl-detox-orpo` is the real multilingual detox model, but it is ~4B params and often OOMs on 14-16 GB GPUs.
# The first two checkpoints are lighter valid detox baselines from the same model collection; they are not fully multilingual,
# but they keep the experimental pipeline runnable. `google/mt5-small` is a generic multilingual seq2seq fallback.
PRETRAINED_DETOX_MODEL_CANDIDATES = [
    "s-nlp/bart-base-detox",
    "s-nlp/ruT5-base-detox",
    "google/mt5-small",
    "s-nlp/mt0-xl-detox-orpo",
]
PRETRAINED_DETOX_MODEL = PRETRAINED_DETOX_MODEL_CANDIDATES[0]

SENTENCE_EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TOXICITY_CLASSIFIER_MODEL = "unitary/multilingual-toxic-xlm-roberta"  # configurable; fallback proxy is used if unavailable

MAX_INPUT_LENGTH = 96
MAX_TARGET_LENGTH = 96

# Safer defaults for 14-16 GB GPUs.
BATCH_SIZE = 1
GENERATION_NUM_BEAMS = 1
RERANK_NUM_BEAMS = 3
RERANK_NUM_RETURN_SEQUENCES = 3

# If bitsandbytes is available, large pretrained seq2seq models can be loaded in 8-bit.
TRY_8BIT_LOADING = True

RUN_PRETRAINED_INFERENCE = True
RUN_TRAINING = True     # Set True to run LoRA/full fine-tuning on GPU.
RUN_RERANKING = True

OUTPUT_DIR = Path("outputs_text_detox_project")
OUTPUT_DIR.mkdir(exist_ok=True)


### Memory note

The original detox checkpoint `s-nlp/mt0-xl-detox-orpo` can exceed 14–16 GB GPU memory. This notebook uses memory-safe defaults: batch size 1, greedy generation by default, valid lighter detox candidates first, optional 8-bit loading through `BitsAndBytesConfig` when supported, and CUDA cache cleanup. For larger GPUs, increase `BATCH_SIZE`, `GENERATION_NUM_BEAMS`, and move `s-nlp/mt0-xl-detox-orpo` earlier in `PRETRAINED_DETOX_MODEL_CANDIDATES`.


## 2. Data loading

In [8]:
def detect_first_existing_column(df: pd.DataFrame, candidates: List[str], required: bool = True) -> Optional[str]:
    lowered = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lowered:
            return lowered[name.lower()]
    if required:
        raise ValueError(f"Cannot detect column. Available columns: {df.columns.tolist()}")
    return None


def read_local_table(path: str) -> pd.DataFrame:
    path = str(path)
    if path.endswith(".csv"):
        return pd.read_csv(path)
    if path.endswith(".tsv") or path.endswith(".txt"):
        return pd.read_csv(path, sep="\t")
    if path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    if path.endswith(".json"):
        return pd.read_json(path)
    raise ValueError(f"Unsupported file extension: {path}")


def load_multilingual_detox_data(local_path: Optional[str] = None) -> pd.DataFrame:
    """Load a multilingual detox dataset from a local file or Hugging Face.

    The loader is intentionally defensive because public dataset schemas may differ.
    If a dataset split has multiple language subsets, they are concatenated with a `lang` column.
    """
    if local_path is not None:
        df = read_local_table(local_path)
        print(f"Loaded local data: {local_path} | shape={df.shape}")
        return df

    try:
        from datasets import load_dataset
    except Exception as e:
        raise ImportError("Install `datasets` or provide LOCAL_DATA_PATH.") from e

    last_error = None
    for dataset_name in DATASET_CANDIDATES:
        try:
            ds = load_dataset(dataset_name)
            frames = []
            for split_name in ds.keys():
                part = ds[split_name].to_pandas()
                # If the split name looks like a language and no language column exists, add it.
                if not any(c in part.columns for c in LANG_COL_CANDIDATES):
                    part["lang"] = split_name
                else:
                    part["split"] = split_name
                frames.append(part)
            df = pd.concat(frames, ignore_index=True)
            print(f"Loaded HF dataset: {dataset_name} | shape={df.shape}")
            return df
        except Exception as e:
            last_error = e
            print(f"Could not load {dataset_name}: {type(e).__name__}: {e}")
    raise RuntimeError("Could not load any dataset candidate.") from last_error


df_raw = load_multilingual_detox_data(LOCAL_DATA_PATH)
print(df_raw.shape)
print(df_raw.columns.tolist())
df_raw.head()

Loaded HF dataset: textdetox/multilingual_paradetox | shape=(3600, 3)
(3600, 3)
['toxic_sentence', 'neutral_sentence', 'lang']


,toxic_sentence,neutral_sentence,lang
0,"then all of a sudden i see her , shes now got ...","All of a sudden i see her, she is all grown up.",en
1,My page should be protected first so that wort...,My page should be protected first so that unpl...,en
2,You made a mistake you ass.,You made a mistake.,en
3,"you know more than these idiots , stay the cou...","you know more than these people , stay the cou...",en
4,"piss me off , fuckin jerk , get on my nerves .",get on my nerves,en


In [9]:
TEXT_COL = detect_first_existing_column(df_raw, TEXT_COL_CANDIDATES, required=True)
REF_COL = detect_first_existing_column(df_raw, REF_COL_CANDIDATES, required=False)
LANG_COL = detect_first_existing_column(df_raw, LANG_COL_CANDIDATES, required=False)

if LANG_COL is None:
    df_raw["lang"] = "unknown"
    LANG_COL = "lang"

print("TEXT_COL:", TEXT_COL)
print("REF_COL:", REF_COL)
print("LANG_COL:", LANG_COL)

# Normalize schema for the rest of the notebook.
df = df_raw.copy()
df["toxic_input"] = df[TEXT_COL].fillna("").astype(str)
df["lang"] = df[LANG_COL].fillna("unknown").astype(str).str.lower()
if REF_COL is not None:
    df["neutral_reference"] = df[REF_COL].fillna("").astype(str)
else:
    df["neutral_reference"] = ""

# Keep only non-empty inputs.
df = df[df["toxic_input"].str.strip().astype(bool)].reset_index(drop=True)
print(df.shape)
df[["lang", "toxic_input", "neutral_reference"]].head()

TEXT_COL: toxic_sentence
REF_COL: neutral_sentence
LANG_COL: lang
(3600, 5)


,lang,toxic_input,neutral_reference
0,en,"then all of a sudden i see her , shes now got ...","All of a sudden i see her, she is all grown up."
1,en,My page should be protected first so that wort...,My page should be protected first so that unpl...
2,en,You made a mistake you ass.,You made a mistake.
3,en,"you know more than these idiots , stay the cou...","you know more than these people , stay the cou..."
4,en,"piss me off , fuckin jerk , get on my nerves .",get on my nerves


## 3. Exploratory data analysis

In [10]:
def simple_token_count(text: str) -> int:
    return len(str(text).split())

eda = df.copy()
eda["input_chars"] = eda["toxic_input"].str.len()
eda["input_tokens"] = eda["toxic_input"].apply(simple_token_count)
eda["has_reference"] = eda["neutral_reference"].str.strip().astype(bool)

print("Rows:", len(eda))
print("Languages:", eda["lang"].nunique())
print("Rows with references:", int(eda["has_reference"].sum()))

display(eda.groupby("lang").agg(
    n=("toxic_input", "size"),
    avg_chars=("input_chars", "mean"),
    avg_tokens=("input_tokens", "mean"),
    references=("has_reference", "sum"),
).sort_values("n", ascending=False).head(30))

Rows: 3600
Languages: 9
Rows with references: 3600


,n,avg_chars,avg_tokens,references
lang,,,,
am,400,70.0400,14.7425,400
ar,400,59.0300,11.2450,400
de,400,105.6450,15.6675,400
en,400,58.3625,11.9575,400
es,400,69.8525,12.2025,400
hi,400,54.4425,11.7600,400
ru,400,64.5250,10.4925,400
uk,400,53.9350,9.2850,400
zh,400,29.5675,1.0525,400


In [11]:
# Show examples per language.
for lang, part in df.groupby("lang"):
    print("=" * 90)
    print("LANG:", lang, "| rows:", len(part))
    sample = part.sample(min(3, len(part)), random_state=42)
    for _, row in sample.iterrows():
        print("TOXIC:", row["toxic_input"][:300])
        if row["neutral_reference"].strip():
            print("REF:  ", row["neutral_reference"][:300])
        print()

LANG: am | rows: 400
TOXIC: ምድረ ደነዝ ሁሉ ጭንቅላቱ ሲጠብ ጭንቅላት ያላቸውን ሰዎች መደብደብ ያው ዝቅተኛነታቸውን ያሳያል:: ይኸው ነው የነታከለ ኡማና ሽመልስ ማፊያ ማደራጀት::
REF:   መሰብ አይችሉም  ሰዎች መደብደብ ያው የበታችነት ስሜት እንደ ሚሰማቸው ያሳያል::

TOXIC: ኮሽ? ምን ዓይነት ርካሽ ስብዕና ነው ያለህ? የኔ ወንድም እህት ጭፍጨፋ . የእናቶቼ ለቅሶ ላንተ ከሆነ ኮሽ . ደህና ቀንህን ጠ
REF:   ኮሽ? እንዴት ነው ነገሩን የምታቀለው አንተ ፡ ወንድመና እህቶቻችን በሞት እያጣን እናቶቻችን እያለቀሱ እኮ ነው !

TOXIC: በነፈሰበት አትንፈስ . ራስህን ሁን አቋም????
REF:   የራስህ አቋም ሰኖርህ ነው የተሻለ የሚሆነው

LANG: ar | rows: 400
TOXIC: دي قلة ادب جديدة ولا شنو ؟؟ لعنة الله عليكم اجمعين ما تسوقون بهكذا افكار وأساليب تخدم اجندة بنو علمان واشباه الرجال ..
REF:   دي قلة ادب جديدة ولا شنو ؟؟ ما تسوقون بهكذا افكار وأساليب تخدم اجندة بنو علمان واشباه الرجال ..

TOXIC: احمد ربك انا اصحى من النوم المطر يخلص الله يلعنك 💔
REF:   احمد ربك انا اصحى من النوم المطر يخلص الله عليك 💔

TOXIC: قسم بالله اغبى قرار تحكيمي شفته بحياتي يلعن ابوك ياكلب
REF:   قسم بالله اغبى قرار تحكيمي شفته بحياتي

LANG: de | rows: 400
TOXIC: Es wird Zeit,den Pfaffen zu zeigen,wer sie sind! Ein Verein der Ges

## 4. Train / validation split

If reference detoxified sentences are available, we split the data into train and validation parts. If no references are available, we keep the full data for inference-only analysis.

In [12]:
from sklearn.model_selection import train_test_split

has_refs = df["neutral_reference"].str.strip().astype(bool)
paired_df = df[has_refs].reset_index(drop=True)
inference_only_df = df[~has_refs].reset_index(drop=True)

if len(paired_df) >= 20:
    train_df, val_df = train_test_split(
        paired_df,
        test_size=0.2,
        random_state=42,
        stratify=paired_df["lang"] if paired_df["lang"].value_counts().min() >= 2 else None,
    )
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
else:
    train_df = paired_df.copy()
    val_df = paired_df.copy()

print("Paired rows:", len(paired_df))
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Inference-only rows:", len(inference_only_df))

Paired rows: 3600
Train rows: 2880
Validation rows: 720
Inference-only rows: 0


## 5. Baseline 1 — copy input

The simplest baseline outputs the toxic input unchanged. It should preserve meaning perfectly but does not reduce toxicity.

In [13]:
df["copy_baseline"] = df["toxic_input"]
if len(val_df):
    val_df = val_df.copy()
    val_df["copy_baseline"] = val_df["toxic_input"]

## 6. Baseline 2 — multilingual rule-based detoxification

This baseline removes or softly replaces toxic words using language-specific lexicons. It is interpretable and fast, but often hurts grammar and meaning.

In [14]:
def load_toxic_lexicon_by_lang(hf_dataset_name: str = "textdetox/multilingual_toxic_lexicon") -> Dict[str, set]:
    try:
        from datasets import load_dataset
        ds = load_dataset(hf_dataset_name)
        lexicons = {}
        for lang in ds.keys():
            part = ds[lang]
            # Most versions use a `text` field.
            col = "text" if "text" in part.column_names else part.column_names[0]
            lexicons[lang.lower()] = {str(w).lower().strip() for w in part[col] if str(w).strip()}
        print("Loaded toxic lexicons:", {k: len(v) for k, v in lexicons.items()})
        return lexicons
    except Exception as e:
        print("Could not load HF toxic lexicon, using compact fallback lexicon.")
        print(type(e).__name__, e)
        return {
            "en": {"idiot", "stupid", "moron", "dumb", "shit", "bullshit", "asshole", "bitch", "crap", "fuck", "fucking"},
            "ru": {"идиот", "идиотка", "дурак", "дура", "дебил", "тупой", "тупая", "тварь", "сука", "блять", "блядь"},
            "uk": {"ідіот", "дурень", "дебіл", "тупий", "сука", "блядь"},
            "es": {"idiota", "estúpido", "estupido", "mierda", "imbécil", "imbecil"},
            "de": {"idiot", "dumm", "scheisse", "scheiße"},
        }

stopwords_by_lang = load_toxic_lexicon_by_lang()

Loaded toxic lexicons: {'am': 245, 'es': 1195, 'ru': 140516, 'uk': 7356, 'en': 3343, 'zh': 3601, 'ar': 430, 'hi': 133, 'de': 247, 'it': 815, 'fr': 1278, 'he': 710, 'hin': 206, 'tt': 15629, 'ja': 328}


In [15]:
soft_replacements = {
    "en": {"idiot": "person", "stupid": "unwise", "moron": "person", "dumb": "unwise", "shit": "bad", "bullshit": "nonsense", "asshole": "rude person", "bitch": "person", "crap": "bad", "fuck": "", "fucking": ""},
    "ru": {"идиот": "человек", "идиотка": "человек", "дурак": "человек", "дура": "человек", "дебил": "человек", "тупой": "неудачный", "тупая": "неудачная", "тварь": "человек", "сука": "человек", "блять": "", "блядь": ""},
    "uk": {"ідіот": "людина", "дурень": "людина", "дебіл": "людина", "тупий": "невдалий", "сука": "людина", "блядь": ""},
    "es": {"idiota": "persona", "estúpido": "poco amable", "estupido": "poco amable", "mierda": "malo", "imbécil": "persona", "imbecil": "persona"},
    "de": {"idiot": "person", "dumm": "unklug", "scheisse": "schlecht", "scheiße": "schlecht"},
}

spaces_re = re.compile(r"\s+")
punct_chars = ".,!?;:()[]{}\"'«»“”‘’"

try:
    import jieba
except Exception:
    jieba = None


def clean_text_after_detox(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"([¿¡])\s+", r"\1", text)
    text = re.sub(r"\s+([»”])", r"\1", text)
    text = re.sub(r"([«“])\s+", r"\1", text)
    return text.strip()


def detoxify_rule_based(text: str, language: str) -> str:
    if pd.isna(text):
        return ""
    original = str(text)
    language = str(language).lower()
    replacements = soft_replacements.get(language, {})
    lang_stopwords = stopwords_by_lang.get(language, set())

    if language == "zh" and jieba is not None:
        tokens = list(jieba.cut(original))
        output_tokens = []
        for token in tokens:
            clean_token = token.lower().strip(punct_chars + " ")
            if clean_token in replacements:
                repl = replacements[clean_token]
                if repl:
                    output_tokens.append(repl)
            elif clean_token not in lang_stopwords:
                output_tokens.append(token)
        result = "".join(output_tokens)
    else:
        output_tokens = []
        for token in spaces_re.split(original):
            clean_token = token.lower().strip(punct_chars + " ")
            if clean_token in replacements:
                repl = replacements[clean_token]
                if repl:
                    output_tokens.append(repl)
                continue
            if clean_token in lang_stopwords:
                continue
            output_tokens.append(token)
        result = " ".join(output_tokens)

    result = clean_text_after_detox(result)
    return result if result else original


df["rule_baseline"] = df.progress_apply(lambda r: detoxify_rule_based(r["toxic_input"], r["lang"]), axis=1)
if len(val_df):
    val_df = val_df.copy()
    val_df["rule_baseline"] = val_df.progress_apply(lambda r: detoxify_rule_based(r["toxic_input"], r["lang"]), axis=1)

changed = (df["toxic_input"] != df["rule_baseline"]).sum()
print("Changed by rule baseline:", changed, "/", len(df))
df[["lang", "toxic_input", "rule_baseline"]].head(10)

  0%|          | 0/3600 [00:00<?, ?it/s]

Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.822 seconds.
Prefix dict has been built successfully.


  0%|          | 0/720 [00:00<?, ?it/s]

Changed by rule baseline: 2506 / 3600


,lang,toxic_input,rule_baseline
0,en,"then all of a sudden i see her , shes now got ...","then all of a sudden i see her, shes now got t..."
1,en,My page should be protected first so that wort...,My page should be protected first so that wort...
2,en,You made a mistake you ass.,You made a mistake you
3,en,"you know more than these idiots , stay the cou...","you know more than these, stay the course and ..."
4,en,"piss me off , fuckin jerk , get on my nerves .","me off,, get on my nerves."
5,en,"In addition, I see no sock puppet, jackass.","In addition, I see no sock puppet,"
6,en,who really gives a crap about this ?,who really gives a bad about this?
7,en,the point is that either choice was shit .,the point is that either choice was bad.
8,en,"Are u there dick, wars back on!!!!",Are u there wars back on!!!!
9,en,"fuck minimum security , put him in real prison .","minimum security, put him in real prison."


## 7. Evaluation utilities

The evaluation uses several signals:

1. **Toxicity score** from a multilingual classifier when available; otherwise a lexicon-based proxy.
2. **Semantic similarity** using multilingual sentence embeddings when available; otherwise a token-overlap proxy.
3. **Reference similarity** using chrF/BERTScore-style metrics when references are available.
4. **Fluency proxy**, based on length and repetition. A stronger version can use perplexity from a language model.

In [16]:
# Optional multilingual sentence encoder.
_sentence_model = None

def get_sentence_model():
    global _sentence_model
    if _sentence_model is not None:
        return _sentence_model
    try:
        from sentence_transformers import SentenceTransformer
        _sentence_model = SentenceTransformer(SENTENCE_EMBEDDING_MODEL, device=DEVICE)
        print("Loaded sentence embedding model:", SENTENCE_EMBEDDING_MODEL)
    except Exception as e:
        print("SentenceTransformer unavailable; using token-overlap similarity.")
        print(type(e).__name__, e)
        _sentence_model = False
    return _sentence_model


def token_jaccard(a: str, b: str) -> float:
    aa = set(str(a).lower().split())
    bb = set(str(b).lower().split())
    if not aa and not bb:
        return 1.0
    if not aa or not bb:
        return 0.0
    return len(aa & bb) / max(1, len(aa | bb))


def semantic_similarity_batch(sources: List[str], candidates: List[str], batch_size: int = 64) -> List[float]:
    model = get_sentence_model()
    if model is False:
        return [token_jaccard(a, b) for a, b in zip(sources, candidates)]

    sims = []
    for i in range(0, len(sources), batch_size):
        src = sources[i:i+batch_size]
        cand = candidates[i:i+batch_size]
        emb_a = model.encode(src, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
        emb_b = model.encode(cand, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
        batch_sims = (emb_a * emb_b).sum(dim=1).detach().cpu().numpy().tolist()
        sims.extend(batch_sims)
    return sims

In [17]:
# Optional toxicity classifier. If unavailable, use lexicon proxy.
_toxicity_pipe = None

def lexicon_toxicity_proxy(text: str, lang: str) -> float:
    words = stopwords_by_lang.get(str(lang).lower(), set())
    if not words:
        words = set().union(*stopwords_by_lang.values()) if stopwords_by_lang else set()
    text_l = str(text).lower()
    if not text_l.strip():
        return 0.0
    hits = 0
    total = max(1, len(text_l.split()))
    for w in words:
        if w and re.search(rf"(?<!\w){re.escape(w)}(?!\w)", text_l):
            hits += 1
    # Smooth bounded proxy.
    return min(1.0, hits / max(1, total) * 4.0)


def get_toxicity_pipeline():
    global _toxicity_pipe
    if _toxicity_pipe is not None:
        return _toxicity_pipe
    try:
        from transformers import pipeline
        _toxicity_pipe = pipeline(
            "text-classification",
            model=TOXICITY_CLASSIFIER_MODEL,
            tokenizer=TOXICITY_CLASSIFIER_MODEL,
            device=0 if DEVICE == "cuda" else -1,
            truncation=True,
            top_k=None,
        )
        print("Loaded toxicity classifier:", TOXICITY_CLASSIFIER_MODEL)
    except Exception as e:
        print("Toxicity classifier unavailable; using lexicon toxicity proxy.")
        print(type(e).__name__, e)
        _toxicity_pipe = False
    return _toxicity_pipe


def parse_toxicity_output(output: Any) -> float:
    """Convert common Hugging Face classifier outputs to a toxic probability."""
    if isinstance(output, list) and output and isinstance(output[0], list):
        output = output[0]
    if isinstance(output, dict):
        output = [output]
    if not isinstance(output, list):
        return np.nan

    best_toxic = 0.0
    for item in output:
        label = str(item.get("label", "")).lower()
        score = float(item.get("score", 0.0))
        if any(key in label for key in ["toxic", "toxicity", "insult", "obscene", "threat", "identity_hate", "label_1", "1"]):
            best_toxic = max(best_toxic, score)
        elif any(key in label for key in ["neutral", "non-toxic", "non_toxic", "label_0", "0"]):
            best_toxic = max(best_toxic, 1.0 - score)
    return float(best_toxic)


def toxicity_scores(texts: List[str], langs: List[str], batch_size: int = 32) -> List[float]:
    pipe = get_toxicity_pipeline()
    if pipe is False:
        return [lexicon_toxicity_proxy(t, l) for t, l in zip(texts, langs)]

    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        try:
            outs = pipe(batch, truncation=True)
            scores.extend([parse_toxicity_output(o) for o in outs])
        except Exception:
            scores.extend([lexicon_toxicity_proxy(t, l) for t, l in zip(batch, langs[i:i+batch_size])])
    return scores

In [18]:
def repetition_rate(text: str, n: int = 3) -> float:
    toks = str(text).lower().split()
    if len(toks) < n:
        return 0.0
    ngrams = [tuple(toks[i:i+n]) for i in range(len(toks)-n+1)]
    if not ngrams:
        return 0.0
    return 1.0 - len(set(ngrams)) / len(ngrams)


def length_ratio_score(source: str, candidate: str) -> float:
    src_len = max(1, len(str(source).split()))
    cand_len = max(1, len(str(candidate).split()))
    ratio = cand_len / src_len
    # Score is highest near ratio 1, lower for overly short/long outputs.
    return float(np.exp(-abs(np.log(ratio))))


def fluency_proxy(source: str, candidate: str) -> float:
    if not str(candidate).strip():
        return 0.0
    return max(0.0, length_ratio_score(source, candidate) - 0.5 * repetition_rate(candidate))


def chrf_scores(references: List[str], predictions: List[str]) -> List[float]:
    try:
        import evaluate
        chrf = evaluate.load("chrf")
        scores = []
        for ref, pred in zip(references, predictions):
            if not str(ref).strip():
                scores.append(np.nan)
            else:
                result = chrf.compute(predictions=[pred], references=[[ref]])
                scores.append(result.get("score", np.nan) / 100.0)
        return scores
    except Exception as e:
        print("chrF unavailable; using semantic similarity to reference as fallback.")
        print(type(e).__name__, e)
        return semantic_similarity_batch(references, predictions)


def evaluate_outputs(frame: pd.DataFrame, output_col: str, name: str = "model") -> pd.DataFrame:
    assert output_col in frame.columns, f"Missing column {output_col}"
    out = frame[["lang", "toxic_input", "neutral_reference", output_col]].copy()
    out = out.rename(columns={output_col: "prediction"})
    out["prediction"] = out["prediction"].fillna("").astype(str)

    out["toxicity_input"] = toxicity_scores(out["toxic_input"].tolist(), out["lang"].tolist())
    out["toxicity_output"] = toxicity_scores(out["prediction"].tolist(), out["lang"].tolist())
    out["toxicity_reduction"] = out["toxicity_input"] - out["toxicity_output"]
    out["meaning_similarity"] = semantic_similarity_batch(out["toxic_input"].tolist(), out["prediction"].tolist())
    out["fluency_proxy"] = [fluency_proxy(a, b) for a, b in zip(out["toxic_input"], out["prediction"])]

    has_ref_mask = out["neutral_reference"].str.strip().astype(bool)
    out["reference_similarity"] = np.nan
    if has_ref_mask.any():
        out.loc[has_ref_mask, "reference_similarity"] = chrf_scores(
            out.loc[has_ref_mask, "neutral_reference"].tolist(),
            out.loc[has_ref_mask, "prediction"].tolist(),
        )

    out["method"] = name
    return out


def summarize_eval(eval_df: pd.DataFrame) -> pd.DataFrame:
    cols = ["toxicity_input", "toxicity_output", "toxicity_reduction", "meaning_similarity", "fluency_proxy", "reference_similarity"]
    return eval_df.groupby("method")[cols].mean(numeric_only=True).sort_values("toxicity_reduction", ascending=False)

## 8. Evaluate simple baselines

In [19]:
eval_frame = val_df if len(val_df) else df.sample(min(200, len(df)), random_state=42).copy()
if "copy_baseline" not in eval_frame.columns:
    eval_frame["copy_baseline"] = eval_frame["toxic_input"]
if "rule_baseline" not in eval_frame.columns:
    eval_frame["rule_baseline"] = eval_frame.progress_apply(lambda r: detoxify_rule_based(r["toxic_input"], r["lang"]), axis=1)

copy_eval = evaluate_outputs(eval_frame, "copy_baseline", "copy")
rule_eval = evaluate_outputs(eval_frame, "rule_baseline", "rule_based")
baseline_eval = pd.concat([copy_eval, rule_eval], ignore_index=True)
summarize_eval(baseline_eval)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: unitary/multilingual-toxic-xlm-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded toxicity classifier: unitary/multilingual-toxic-xlm-roberta


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded sentence embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


,toxicity_input,toxicity_output,toxicity_reduction,meaning_similarity,fluency_proxy,reference_similarity
method,,,,,,
rule_based,0.328211,0.165525,0.162687,0.955676,0.931503,0.687333
copy,0.328211,0.328211,0.000000,1.000000,0.999957,0.680835


In [20]:
# Per-language diagnostics for the rule baseline.
rule_eval.groupby("lang")[["toxicity_reduction", "meaning_similarity", "fluency_proxy", "reference_similarity"]].mean(numeric_only=True).sort_values("toxicity_reduction", ascending=False).head(30)

,toxicity_reduction,meaning_similarity,fluency_proxy,reference_similarity
lang,,,,
en,0.739211,0.917522,0.851661,0.701139
es,0.259450,0.972230,0.953047,0.686729
uk,0.203715,0.940034,0.870678,0.782644
ru,0.176060,0.965852,0.922550,0.698344
zh,0.035680,0.888689,1.000000,0.519750
de,0.023973,0.980568,0.957868,0.834345
am,0.023754,0.988077,0.949630,0.440164
hi,0.001424,0.968639,0.949319,0.742982
ar,0.000914,0.979476,0.928774,0.779900


## 9. Baseline 3 — pretrained seq2seq detoxification model

This section applies a pretrained multilingual detoxification model. It is an inference baseline, not our final research contribution. It helps compare fine-tuning and reranking against a strong existing model.

In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

seq2seq_tokenizer = None
seq2seq_model = None
loaded_pretrained_model_name = None

lang_prompts = {
    "zh": "排毒：",
    "es": "Desintoxicar: ",
    "ru": "Детоксифицируй: ",
    "ar": "إزالة السموم: ",
    "hi": "विषहरण: ",
    "uk": "Детоксифікуй: ",
    "de": "Entgiften: ",
    "am": "Detoxify: ",
    "en": "Detoxify: ",
}

def build_prompt(text: str, lang: str) -> str:
    return lang_prompts.get(str(lang).lower(), "Detoxify: ") + str(text)


def _has_bitsandbytes() -> bool:
    try:
        import bitsandbytes  # noqa: F401
        return True
    except Exception:
        return False


def _model_device(model):
    """Return a safe device for token tensors."""
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device(DEVICE)


def unload_pretrained_detox_model():
    """Free the currently loaded seq2seq baseline model."""
    global seq2seq_tokenizer, seq2seq_model, loaded_pretrained_model_name
    try:
        del seq2seq_model
        del seq2seq_tokenizer
    except Exception:
        pass
    seq2seq_tokenizer = None
    seq2seq_model = None
    loaded_pretrained_model_name = None
    clear_cuda_memory()


def _load_regular_seq2seq(model_name: str, config):
    """Load a regular fp16/fp32 seq2seq model without quantization."""
    kwargs = {"config": config}
    if DEVICE == "cuda":
        kwargs.update({"torch_dtype": torch.float16, "low_cpu_mem_usage": True})
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **kwargs).to(DEVICE)
    else:
        kwargs.update({"torch_dtype": torch.float32})
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **kwargs).to(DEVICE)
    return model


def _load_quantized_seq2seq(model_name: str, config):
    """Load an 8-bit seq2seq model using the modern quantization_config API.

    Older notebooks often use `load_in_8bit=True` directly. In some transformer
    versions this argument is passed into the model constructor and causes:
    `__init__() got an unexpected keyword argument 'load_in_8bit'`.
    `BitsAndBytesConfig` is the safer modern API. If unsupported, caller falls
    back to regular loading.
    """
    if DEVICE != "cuda" or not TRY_8BIT_LOADING or BitsAndBytesConfig is None or not _has_bitsandbytes():
        raise RuntimeError("8-bit loading is unavailable in this environment.")

    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    return AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        config=config,
        device_map="auto",
        quantization_config=quant_config,
        low_cpu_mem_usage=True,
    )


def load_pretrained_detox_model(model_candidates=None, prefer_8bit: bool = TRY_8BIT_LOADING):
    """Load the first seq2seq baseline that works in the current environment.

    Notes:
    - `s-nlp/mt0-small-detox-orpo` and `s-nlp/mt0-base-detox-orpo` are not public valid HF ids.
    - `s-nlp/mt0-xl-detox-orpo` is valid, but very large (~4B), so it is tried last by default.
    - If 8-bit quantization fails because of transformers/bitsandbytes versions, the loader retries
      the same checkpoint without quantization instead of stopping the notebook.
    """
    global seq2seq_tokenizer, seq2seq_model, loaded_pretrained_model_name

    if seq2seq_model is not None:
        return seq2seq_tokenizer, seq2seq_model

    if model_candidates is None:
        if isinstance(PRETRAINED_DETOX_MODEL_CANDIDATES, (list, tuple)):
            model_candidates = list(PRETRAINED_DETOX_MODEL_CANDIDATES)
        else:
            model_candidates = [PRETRAINED_DETOX_MODEL]

    last_error = None

    for model_name in model_candidates:
        print(f"\nTrying to load seq2seq baseline: {model_name}")
        unload_pretrained_detox_model()

        try:
            config = AutoConfig.from_pretrained(model_name)
            # Some checkpoints store both shared and lm_head weights. This silences tied-weight warnings.
            if hasattr(config, "tie_word_embeddings"):
                config.tie_word_embeddings = False

            tokenizer = AutoTokenizer.from_pretrained(model_name)

            model = None
            if prefer_8bit and DEVICE == "cuda":
                try:
                    model = _load_quantized_seq2seq(model_name, config)
                    print("Loaded with 8-bit quantization.")
                except Exception as qe:
                    print(f"8-bit loading failed for {model_name}; retrying regular loading. ({type(qe).__name__}: {qe})")
                    clear_cuda_memory()

            if model is None:
                model = _load_regular_seq2seq(model_name, config)

            model.eval()
            seq2seq_tokenizer = tokenizer
            seq2seq_model = model
            loaded_pretrained_model_name = model_name
            print("Loaded pretrained seq2seq model:", model_name)
            return seq2seq_tokenizer, seq2seq_model

        except torch.cuda.OutOfMemoryError as e:
            last_error = e
            print(f"CUDA OOM for {model_name}. Trying the next/lighter option...")
            unload_pretrained_detox_model()
            continue
        except Exception as e:
            last_error = e
            print(f"Could not load {model_name}: {type(e).__name__}: {e}")
            unload_pretrained_detox_model()
            continue

    raise RuntimeError(f"No pretrained seq2seq baseline could be loaded. Last error: {repr(last_error)}")


def generate_seq2seq_outputs(
    texts: List[str],
    langs: List[str],
    tokenizer,
    model,
    batch_size: int = BATCH_SIZE,
    num_beams: int = GENERATION_NUM_BEAMS,
    num_return_sequences: int = 1,
    do_sample: bool = False,
    temperature: float = 0.9,
) -> List[Any]:
    """Memory-safe seq2seq generation.

    If CUDA OOM happens during generation, the function retries the same examples
    one-by-one with greedy decoding.
    """
    all_outputs = []
    effective_batch_size = max(1, int(batch_size))
    tensor_device = _model_device(model)

    for i in tqdm(range(0, len(texts), effective_batch_size), desc="seq2seq generation"):
        batch_texts = texts[i:i+effective_batch_size]
        batch_langs = langs[i:i+effective_batch_size]
        prompts = [build_prompt(t, l) for t, l in zip(batch_texts, batch_langs)]

        try:
            enc = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_LENGTH,
            )
            enc = {k: v.to(tensor_device) for k, v in enc.items()}

            gen_kwargs = dict(
                **enc,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=num_beams,
                num_return_sequences=num_return_sequences,
                do_sample=do_sample,
                no_repeat_ngram_size=3,
                repetition_penalty=1.15,
                early_stopping=True,
            )
            if do_sample:
                gen_kwargs["temperature"] = temperature

            with torch.inference_mode():
                out = model.generate(**gen_kwargs)

            decoded = tokenizer.batch_decode(out, skip_special_tokens=True)

            if num_return_sequences == 1:
                all_outputs.extend([clean_text_after_detox(x) for x in decoded])
            else:
                grouped = []
                for j in range(0, len(decoded), num_return_sequences):
                    grouped.append([clean_text_after_detox(x) for x in decoded[j:j+num_return_sequences]])
                all_outputs.extend(grouped)

        except torch.cuda.OutOfMemoryError:
            print("CUDA OOM during generation. Retrying this batch item-by-item with greedy decoding.")
            clear_cuda_memory()
            if effective_batch_size == 1 and num_beams == 1:
                raise

            retry_outputs = generate_seq2seq_outputs(
                batch_texts,
                batch_langs,
                tokenizer,
                model,
                batch_size=1,
                num_beams=1,
                num_return_sequences=1,
                do_sample=False,
            )
            if num_return_sequences == 1:
                all_outputs.extend(retry_outputs)
            else:
                all_outputs.extend([[x] for x in retry_outputs])

    clear_cuda_memory()
    return all_outputs


In [22]:
if RUN_PRETRAINED_INFERENCE:
    try:
        tokenizer_detox, model_detox = load_pretrained_detox_model()
        eval_frame = eval_frame.copy()
        eval_frame["pretrained_detox"] = generate_seq2seq_outputs(
            eval_frame["toxic_input"].tolist(),
            eval_frame["lang"].tolist(),
            tokenizer_detox,
            model_detox,
            batch_size=BATCH_SIZE,
            num_beams=GENERATION_NUM_BEAMS,
            num_return_sequences=1,
        )
        pretrained_eval = evaluate_outputs(eval_frame, "pretrained_detox", "pretrained_seq2seq")
        all_eval = pd.concat([baseline_eval, pretrained_eval], ignore_index=True)
        print("Pretrained model used:", loaded_pretrained_model_name)
        display(summarize_eval(all_eval))
    except Exception as e:
        print("Pretrained inference was skipped because no model could be loaded/run safely.")
        print(type(e).__name__, e)
        all_eval = baseline_eval.copy()
else:
    all_eval = baseline_eval.copy()



Trying to load seq2seq baseline: s-nlp/bart-base-detox


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/295 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loaded with 8-bit quantization.
Loaded pretrained seq2seq model: s-nlp/bart-base-detox


seq2seq generation:   0%|          | 0/720 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Pretrained model used: s-nlp/bart-base-detox


,toxicity_input,toxicity_output,toxicity_reduction,meaning_similarity,fluency_proxy,reference_similarity
method,,,,,,
pretrained_seq2seq,0.328211,0.002400,0.325811,0.136112,0.161205,0.034176
rule_based,0.328211,0.165525,0.162687,0.955676,0.931503,0.687333
copy,0.328211,0.328211,0.000000,1.000000,0.999957,0.680835


## 10. Main model — optional mT5 fine-tuning / LoRA fine-tuning

This is the core deep learning extension. The model learns directly from toxic-neutral sentence pairs.

Recommended setup:
- Base model: `google/mt5-small`.
- Input format: `detoxify <lang>: <toxic sentence>`.
- Target: neutral sentence.
- Training objective: token-level cross-entropy with teacher forcing.
- Efficient option: LoRA adapters over attention projections.

Set `RUN_TRAINING = True` in the config section to run this block.

In [23]:
def make_training_text(row: pd.Series) -> str:
    return f"detoxify {row['lang']}: {row['toxic_input']}"


def tokenize_seq2seq_batch(batch, tokenizer):
    inputs = [f"detoxify {l}: {x}" for x, l in zip(batch["toxic_input"], batch["lang"])]
    targets = batch["neutral_reference"]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


def prepare_hf_dataset(train_frame: pd.DataFrame, valid_frame: pd.DataFrame):
    from datasets import Dataset
    train_ds = Dataset.from_pandas(train_frame[["toxic_input", "neutral_reference", "lang"]].reset_index(drop=True))
    valid_ds = Dataset.from_pandas(valid_frame[["toxic_input", "neutral_reference", "lang"]].reset_index(drop=True))
    return train_ds, valid_ds

In [25]:
if RUN_TRAINING:
    assert len(train_df) > 0 and len(val_df) > 0, "Fine-tuning requires paired toxic-neutral data."

    import inspect
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForSeq2SeqLM,
        DataCollatorForSeq2Seq,
        Seq2SeqTrainer,
        Seq2SeqTrainingArguments,
    )

    clear_cuda_memory()

    ft_tokenizer = AutoTokenizer.from_pretrained(BASE_SEQ2SEQ_MODEL)

    # Memory-safe model loading.
    model_kwargs = {}
    if DEVICE == "cuda":
        model_kwargs["torch_dtype"] = torch.float16

    ft_model = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_SEQ2SEQ_MODEL,
        **model_kwargs,
    )

    # This silences tied-weights warnings for some T5/mT5 checkpoints.
    if hasattr(ft_model.config, "tie_word_embeddings"):
        ft_model.config.tie_word_embeddings = False

    ft_model.to(DEVICE)

    USE_LORA = True

    if USE_LORA:
        try:
            from peft import LoraConfig, TaskType, get_peft_model

            lora_config = LoraConfig(
                task_type=TaskType.SEQ_2_SEQ_LM,
                r=8,
                lora_alpha=16,
                lora_dropout=0.05,
                target_modules=["q", "v"],
            )

            ft_model = get_peft_model(ft_model, lora_config)
            ft_model.print_trainable_parameters()

        except Exception as e:
            print("PEFT/LoRA unavailable. Falling back to full fine-tuning.")
            print(type(e).__name__, e)

    hf_train, hf_val = prepare_hf_dataset(train_df, val_df)

    hf_train = hf_train.map(
        lambda batch: tokenize_seq2seq_batch(batch, ft_tokenizer),
        batched=True,
        remove_columns=hf_train.column_names,
    )

    hf_val = hf_val.map(
        lambda batch: tokenize_seq2seq_batch(batch, ft_tokenizer),
        batched=True,
        remove_columns=hf_val.column_names,
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=ft_tokenizer,
        model=ft_model,
    )

    # Different transformers versions use different argument names:
    # old: evaluation_strategy
    # new: eval_strategy
    training_args_signature = inspect.signature(Seq2SeqTrainingArguments.__init__)
    strategy_arg_name = (
        "eval_strategy"
        if "eval_strategy" in training_args_signature.parameters
        else "evaluation_strategy"
    )

    training_args_kwargs = dict(
        output_dir=str(OUTPUT_DIR / "mt5_detox_lora"),
        eval_steps=250,
        save_steps=250,
        logging_steps=50,
        learning_rate=2e-4,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        predict_with_generate=True,
        fp16=(DEVICE == "cuda"),
        save_total_limit=2,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        generation_max_length=MAX_TARGET_LENGTH,
    )

    training_args_kwargs[strategy_arg_name] = "steps"

    # Some versions require save_strategy to match eval_strategy
    # when load_best_model_at_end=True.
    training_args_signature = inspect.signature(Seq2SeqTrainingArguments.__init__)
    if "save_strategy" in training_args_signature.parameters:
        training_args_kwargs["save_strategy"] = "steps"

    # Some versions do not support generation_max_length.
    if "generation_max_length" not in training_args_signature.parameters:
        training_args_kwargs.pop("generation_max_length", None)

    training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

    trainer_kwargs = dict(
        model=ft_model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        data_collator=data_collator,
    )

    # Newer versions deprecate tokenizer= and use processing_class=.
    trainer_signature = inspect.signature(Seq2SeqTrainer.__init__)
    if "processing_class" in trainer_signature.parameters:
        trainer_kwargs["processing_class"] = ft_tokenizer
    else:
        trainer_kwargs["tokenizer"] = ft_tokenizer

    trainer = Seq2SeqTrainer(**trainer_kwargs)

    trainer.train()

    final_model_dir = OUTPUT_DIR / "mt5_detox_lora_final"
    trainer.save_model(str(final_model_dir))
    ft_tokenizer.save_pretrained(str(final_model_dir))

    print(f"Saved fine-tuned model to: {final_model_dir}")

    clear_cuda_memory()

else:
    print("Training is disabled. Set RUN_TRAINING = True to fine-tune mT5/LoRA.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 344,064 || all params: 556,635,520 || trainable%: 0.0618


Map:   0%|          | 0/2880 [00:00<?, ? examples/s]

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
250,204.412187,14.757812


Saved fine-tuned model to: outputs_text_detox_project/mt5_detox_lora_final


In [26]:
# Optional evaluation of a fine-tuned model after training.
if RUN_TRAINING:
    ft_model.eval()
    eval_frame = eval_frame.copy()
    eval_frame["finetuned_mt5"] = generate_seq2seq_outputs(
        eval_frame["toxic_input"].tolist(),
        eval_frame["lang"].tolist(),
        ft_tokenizer,
        ft_model,
        batch_size=BATCH_SIZE,
        num_beams=4,
        num_return_sequences=1,
    )
    ft_eval = evaluate_outputs(eval_frame, "finetuned_mt5", "finetuned_mt5_lora")
    all_eval = pd.concat([all_eval, ft_eval], ignore_index=True)
    display(summarize_eval(all_eval))

seq2seq generation:   0%|          | 0/720 [00:00<?, ?it/s]

,toxicity_input,toxicity_output,toxicity_reduction,meaning_similarity,fluency_proxy,reference_similarity
method,,,,,,
pretrained_seq2seq,0.328211,0.002400,0.325811,0.136112,0.161205,0.034176
finetuned_mt5_lora,0.328211,0.037368,0.290844,0.256348,0.274440,0.100886
rule_based,0.328211,0.165525,0.162687,0.955676,0.931503,0.687333
copy,0.328211,0.328211,0.000000,1.000000,0.999957,0.680835


## 11. Candidate generation and multi-objective reranking

Instead of accepting the first generated sequence, we generate multiple candidates and select the one with the best trade-off:

$$
S(y) = lpha \cdot sim(x,y) + eta \cdot fluency(y) - \gamma \cdot toxicity(y) - \delta \cdot repetition(y)
$$

This explicitly models the detoxification trade-off: the system should reduce toxicity without deleting the meaning.

In [30]:
def bad_output(original: str, candidate: str) -> bool:
    original = str(original).strip()
    candidate = str(candidate).strip()
    if not candidate:
        return True
    if len(candidate) < 2:
        return True
    if len(candidate) < 0.20 * max(1, len(original)):
        return True
    bad_prefixes = ["detoxify:", "detoxify", "детоксифицируй", "desintoxicar", "entgiften", "排毒"]
    if candidate.lower().startswith(tuple(bad_prefixes)):
        return True
    return False


def rerank_candidates_for_frame(
    frame: pd.DataFrame,
    candidates_col: str,
    fallback_col: str = "rule_baseline",
    alpha: float = 0.45,
    beta: float = 0.20,
    gamma: float = 0.30,
    delta: float = 0.05,
) -> Tuple[List[str], pd.DataFrame]:
    final_outputs = []
    details = []

    # Flatten candidates for batch scoring.
    flat_sources, flat_langs, flat_candidates, owner = [], [], [], []
    for idx, row in frame.iterrows():
        candidates = row[candidates_col]
        if not isinstance(candidates, list):
            candidates = [str(candidates)]
        # Always include fallback candidate as a safe option.
        if fallback_col in frame.columns:
            candidates = candidates + [row[fallback_col]]
        for cand in candidates:
            cand = clean_text_after_detox(cand)
            if bad_output(row["toxic_input"], cand):
                continue
            flat_sources.append(row["toxic_input"])
            flat_langs.append(row["lang"])
            flat_candidates.append(cand)
            owner.append(idx)

    if not flat_candidates:
        return frame.get(fallback_col, frame["toxic_input"]).tolist(), pd.DataFrame()

    tox = toxicity_scores(flat_candidates, flat_langs)
    sim = semantic_similarity_batch(flat_sources, flat_candidates)
    flu = [fluency_proxy(s, c) for s, c in zip(flat_sources, flat_candidates)]
    rep = [repetition_rate(c) for c in flat_candidates]
    score = [alpha*s + beta*f - gamma*t - delta*r for s, f, t, r in zip(sim, flu, tox, rep)]

    scored = pd.DataFrame({
        "row_id": owner,
        "candidate": flat_candidates,
        "toxicity": tox,
        "meaning_similarity": sim,
        "fluency_proxy": flu,
        "repetition_rate": rep,
        "rerank_score": score,
    })

    best_by_row = scored.sort_values("rerank_score", ascending=False).groupby("row_id").head(1).set_index("row_id")
    for idx, row in frame.iterrows():
        if idx in best_by_row.index:
            final_outputs.append(best_by_row.loc[idx, "candidate"])
        elif fallback_col in frame.columns:
            final_outputs.append(row[fallback_col])
        else:
            final_outputs.append(row["toxic_input"])
    return final_outputs, scored

In [31]:
if RUN_RERANKING and RUN_PRETRAINED_INFERENCE:
    try:
        tokenizer_detox, model_detox = load_pretrained_detox_model()
        eval_frame = eval_frame.copy()
        if "rule_baseline" not in eval_frame.columns:
            eval_frame["rule_baseline"] = eval_frame.progress_apply(lambda r: detoxify_rule_based(r["toxic_input"], r["lang"]), axis=1)

        eval_frame["pretrained_candidates"] = generate_seq2seq_outputs(
            eval_frame["toxic_input"].tolist(),
            eval_frame["lang"].tolist(),
            tokenizer_detox,
            model_detox,
            batch_size=1,
            num_beams=RERANK_NUM_BEAMS,
            num_return_sequences=RERANK_NUM_RETURN_SEQUENCES,
        )
        eval_frame["reranked_detox"], rerank_details = rerank_candidates_for_frame(
            eval_frame,
            candidates_col="pretrained_candidates",
            fallback_col="rule_baseline",
        )
        reranked_eval = evaluate_outputs(eval_frame, "reranked_detox", "candidate_reranking")
        all_eval = pd.concat([all_eval, reranked_eval], ignore_index=True)
        print("Pretrained model used for reranking:", loaded_pretrained_model_name)
        display(summarize_eval(all_eval))
    except Exception as e:
        print("Reranking was skipped because candidate generation failed.")
        print(type(e).__name__, e)
else:
    print("Reranking skipped.")


seq2seq generation:   0%|          | 0/720 [00:00<?, ?it/s]

Pretrained model used for reranking: s-nlp/bart-base-detox


,toxicity_input,toxicity_output,toxicity_reduction,meaning_similarity,fluency_proxy,reference_similarity
method,,,,,,
pretrained_seq2seq,0.328211,0.002400,0.325811,0.136112,0.161205,0.034176
finetuned_mt5_lora,0.328211,0.037368,0.290844,0.256348,0.274440,0.100886
candidate_reranking,0.328211,0.165525,0.162687,0.955676,0.931503,0.687333
rule_based,0.328211,0.165525,0.162687,0.955676,0.931503,0.687333
copy,0.328211,0.328211,0.000000,1.000000,0.999957,0.680835


## 12. Per-language comparison

In [32]:
def per_language_summary(eval_df: pd.DataFrame) -> pd.DataFrame:
    return eval_df.groupby(["method", "lang"])[[
        "toxicity_output", "toxicity_reduction", "meaning_similarity", "fluency_proxy", "reference_similarity"
    ]].mean(numeric_only=True).reset_index().sort_values(["method", "toxicity_reduction"], ascending=[True, False])

per_lang = per_language_summary(all_eval)
per_lang.head(50)

,method,lang,toxicity_output,toxicity_reduction,meaning_similarity,fluency_proxy,reference_similarity
3,candidate_reranking,en,0.176499,0.739211,0.917522,0.851661,0.701139
4,candidate_reranking,es,0.174708,0.259450,0.972230,0.953047,0.686729
7,candidate_reranking,uk,0.175343,0.203715,0.940034,0.870678,0.782644
6,candidate_reranking,ru,0.185976,0.176060,0.965852,0.922550,0.698344
8,candidate_reranking,zh,0.217155,0.035680,0.888689,1.000000,0.519750
2,candidate_reranking,de,0.155839,0.023973,0.980568,0.957868,0.834345
0,candidate_reranking,am,0.264576,0.023754,0.988077,0.949630,0.440164
5,candidate_reranking,hi,0.075182,0.001424,0.968639,0.949319,0.742982
1,candidate_reranking,ar,0.064444,0.000914,0.979476,0.928774,0.779900
9,copy,am,0.288330,0.000000,1.000000,1.000000,0.438781


In [33]:
# Pivot view for easier comparison.
metric = "toxicity_reduction"
per_lang.pivot_table(index="lang", columns="method", values=metric, aggfunc="mean").sort_index()

method,candidate_reranking,copy,finetuned_mt5_lora,pretrained_seq2seq,rule_based
lang,,,,,
am,0.023754,0.0,0.209026,0.286793,0.023754
ar,0.000914,0.0,0.059120,0.063414,0.000914
de,0.023973,0.0,0.150877,0.178750,0.023973
en,0.739211,0.0,0.855460,0.909539,0.739211
es,0.259450,0.0,0.430604,0.433109,0.259450
hi,0.001424,0.0,0.064689,0.071402,0.001424
ru,0.176060,0.0,0.332985,0.360614,0.176060
uk,0.203715,0.0,0.341876,0.377644,0.203715
zh,0.035680,0.0,0.172956,0.251036,0.035680


## 13. Error analysis

We inspect examples where the system fails. Typical failure categories:

- toxicity remains;
- meaning changed too much;
- output is copied;
- output is too short;
- grammar or language is broken;
- prompt artifacts are generated.

In [34]:
def attach_eval_columns(frame: pd.DataFrame, output_col: str) -> pd.DataFrame:
    e = evaluate_outputs(frame, output_col, output_col)
    result = frame.copy()
    for col in ["toxicity_input", "toxicity_output", "toxicity_reduction", "meaning_similarity", "fluency_proxy", "reference_similarity"]:
        result[col] = e[col].values
    return result

preferred_output_col = None
for col in ["reranked_detox", "finetuned_mt5", "pretrained_detox", "rule_baseline"]:
    if col in eval_frame.columns:
        preferred_output_col = col
        break

print("Preferred output for error analysis:", preferred_output_col)

if preferred_output_col:
    analysis_df = attach_eval_columns(eval_frame, preferred_output_col)
    analysis_df["copied"] = analysis_df["toxic_input"].str.strip() == analysis_df[preferred_output_col].str.strip()
    analysis_df["too_short"] = analysis_df[preferred_output_col].str.len() < 0.25 * analysis_df["toxic_input"].str.len()
    analysis_df["potential_failure_score"] = (
        analysis_df["toxicity_output"].fillna(0)
        + (1 - analysis_df["meaning_similarity"].fillna(0))
        + analysis_df["copied"].astype(float) * 0.5
        + analysis_df["too_short"].astype(float) * 0.5
    )
    display(analysis_df.sort_values("potential_failure_score", ascending=False)[[
        "lang", "toxic_input", "neutral_reference", preferred_output_col,
        "toxicity_output", "toxicity_reduction", "meaning_similarity", "fluency_proxy", "copied", "too_short"
    ]].head(30))

Preferred output for error analysis: reranked_detox


,lang,toxic_input,neutral_reference,reranked_detox,toxicity_output,toxicity_reduction,meaning_similarity,fluency_proxy,copied,too_short
386,en,My page should be protected first so that wort...,My page should be protected first so that unpl...,My page should be protected first so that wort...,0.947376,0.000000,1.000000,1.000000,True,False
121,es,Debería darte vergüenza ofender las creencias ...,Debería darte vergüenza ofender las creencias ...,Debería darte vergüenza ofender las creencias ...,0.915812,0.000000,1.000000,1.000000,True,False
70,en,"priceless, I hate that guy","Priceless, I dislike that guy.","priceless, I hate that guy",0.897663,0.000000,1.000000,1.000000,True,False
601,de,dieser Holtmann scheint ein Grüner-Bastard zu ...,dieser Holtmann scheint ein Grüner-Anhänger zu...,dieser Holtmann scheint ein Grüner-Bastard zu ...,0.856753,0.000000,1.000000,1.000000,True,False
75,ru,"дурочка ты,у тебя мамы нет,ласковое слово жалко!","у тебя мамы нет,ласковое слово жалко!","дурочка ты,у тебя мамы нет,ласковое слово жалко!",0.836655,0.000000,1.000000,1.000000,True,False
266,en,And how can you justify a moronic 2 day block?,And how can you justify a 2 day block?,And how can you justify a moronic 2 day block?,0.807160,0.000000,1.000000,1.000000,True,False
113,uk,"сукаааа, я більше не полечу нікуди!!!!!",я більше не полечу нікуди!!!!!,"сукаааа, я більше не полечу нікуди!!!!!",0.784269,0.000000,1.000000,1.000000,True,False
497,ru,"уже не знают как привлечь к себе внимание, ебё...","Я уже не знают, как обратить на себя внимание,...","уже не знают как привлечь к себе внимание, ебё...",0.764211,0.000000,1.000000,1.000000,True,False
120,ru,Он же официально беженец в гейропке,Он же официально беженец в европе,Он же официально беженец в гейропке,0.639422,0.000000,1.000000,1.000000,True,False
26,es,ya esta el subnormal de luis enrique jodiendom...,ya esta luis enrique arruinandome apuestas sin...,ya esta el subnormal de luis enrique jodiendom...,0.638796,0.000000,1.000000,1.000000,True,False


In [35]:
# Save analysis artifacts
if preferred_output_col:
    result_cols = ["lang", "toxic_input", "neutral_reference", "copy_baseline", "rule_baseline"]
    for col in ["pretrained_detox", "reranked_detox", "finetuned_mt5"]:
        if col in eval_frame.columns:
            result_cols.append(col)
    eval_frame[result_cols].to_csv(OUTPUT_DIR / "model_outputs_for_analysis.csv", index=False)
    all_eval.to_csv(OUTPUT_DIR / "automatic_evaluation_results.csv", index=False)
    per_lang.to_csv(OUTPUT_DIR / "per_language_evaluation.csv", index=False)
    print("Saved analysis files to", OUTPUT_DIR.resolve())

Saved analysis files to /kaggle/working/outputs_text_detox_project


## 14. Discussion template

Use the automatically computed tables above to fill this section after running the notebook.

### Expected observations

1. **Copy baseline** should have high semantic preservation but near-zero toxicity reduction.
2. **Rule-based baseline** should reduce explicit toxic tokens, but often harms grammar and meaning.
3. **Pretrained seq2seq model** should improve fluency and paraphrasing, but may fail on low-resource languages or produce prompt artifacts.
4. **Fine-tuned / LoRA model** should adapt better to the dataset distribution when enough paired examples are available.
5. **Candidate reranking** should improve the balance between detoxification and meaning preservation by rejecting too short, still-toxic, or repetitive candidates.

### Limitations

- Automatic toxicity classifiers can be biased across languages.
- Meaning preservation is difficult to measure for paraphrases.
- Low-resource languages may suffer from tokenizer fragmentation and insufficient training examples.
- Rule-based lexicons miss implicit toxicity and sarcasm.

### Future work

- Add human evaluation for adequacy, fluency, and toxicity.
- Train with a larger multilingual dataset.
- Compare mT5, mBART, and instruction-tuned models.
- Add language-specific adapters.
- Use preference optimization or reward-based reranking for better style control.

## 15. Final project summary

This notebook implements a full deep learning research pipeline for multilingual text detoxification:

- The task is formulated as controlled sequence-to-sequence generation.
- Several baselines are compared: copy, rule-based, pretrained seq2seq, and optional fine-tuned mT5/LoRA.
- Evaluation uses multiple objectives: toxicity reduction, semantic preservation, fluency proxy, and per-language robustness.
- Candidate reranking explicitly optimizes the detoxification trade-off.
- The output is a research analysis, not a competition submission.